# Callosal Angle (CA) — MRI Adaptation

Measures the angle between the medial walls of the lateral ventricles on a coronal slice at the Posterior Commissure (PC) level.

**MRI adaptation**: No HU windowing. Otsu + adaptive thresholding on raw T1w intensities. For OASIS, coronal slices from `arr[:, y, :]`. For ABCD, use `extract_coronal_view()`.

**Reference**: Samadani et al., JNS 2024

## Library imports

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import cv2 as cv
import SimpleITK as sitk
from skimage import measure, filters
import statsmodels.api as sm
import matplotlib.pyplot as plt

from mri_utils import (rotate_image, rotate_image_coronal, getLargestCC, extract_coronal_view,
                       moore_contour, trace_left_contour, trace_right_contour, fit_lines,
                       normalize_to_uint8,
                       OASIS_AC_VOXEL, OASIS_PC_VOXEL, parse_acpc_coords, load_acpc_from_txt)

## Configuration

In [ ]:
# ── Choose your dataset ──────────────────────────────────────────────
DATASET = "oasis1"  # "oasis1", "abcd", or "custom"

# ── Dataset-specific settings ────────────────────────────────────────
if DATASET == "oasis1":
    DATA_DIR = "data/oasis1/disc1"
    DEMOGRAPHICS_PATH = "data/oasis1/oasis_cross-sectional-5708aa0a98d82080.xlsx"
    # OASIS skull-stripped, gain-field corrected, atlas-registered volumes
    SCAN_GLOB = "OAS1_*/PROCESSED/MPRAGE/T88_111/*_masked_gfc.img"
    SUBJECT_ID_FROM_PATH = lambda p: p.split(os.sep)[-5]  # e.g. OAS1_0001_MR1
    # For OASIS: AC/PC are fixed atlas voxel coordinates — no per-subject txt needed
    ACPC_TXT_GLOB = None

elif DATASET == "abcd":
    DATA_DIR = "data/abcd"
    DEMOGRAPHICS_PATH = None  # set path to ABCD demographics CSV/TSV if available
    # ART-corrected T1w scans (output from Charlie's ART batch script)
    SCAN_GLOB = "**/*_T1w_acpc.nii*"
    SUBJECT_ID_FROM_PATH = lambda p: os.path.basename(p).split("_ses")[0]  # e.g. sub-NDARINVXXXXXXXX
    # For ABCD: per-subject AC/PC txt files produced alongside ART-corrected scans
    ACPC_TXT_GLOB = "**/*_acpc_coords.txt"

elif DATASET == "custom":
    DATA_DIR = "data/my_dataset"
    DEMOGRAPHICS_PATH = None
    SCAN_GLOB = "**/*.nii.gz"  # adjust to your file naming
    SUBJECT_ID_FROM_PATH = lambda p: os.path.basename(p).replace(".nii.gz", "").replace(".nii", "")
    ACPC_TXT_GLOB = "**/*_acpc_coords.txt"  # adjust if needed

else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}. Use 'oasis1', 'abcd', or 'custom'.")

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Dataset: {DATASET}")
print(f"Data dir: {DATA_DIR}")
print(f"Scan pattern: {SCAN_GLOB}")

## Processing parameters

In [ ]:
ca_processing_params = {
    "gauss_blur_kernel_size": 3,
    "adaptive_thresh_size": 75,
    "adaptive_thresh_C": 7.5,
    "above_pc_mask_threshold": 10,
    "middle_roi_xlims": 20,
    "middle_roi_upper_ylim": 15,
    "smoothing_factor": 7,
    "dec_smooth_factor_max_thresh": 4,
    "initial_smooth_ven_fullbrain_ratio": 0.008,
    "smoothed_ven_fullbrain_ratio": 0.02,
    "median_blur_kernel_size": 3,
}

## Connected component cleanup

In [ ]:
def num_connected_comps_cv(clean_ventricles, suppress_blob_area_ratio=0.3, disconnectedLV_height_diff_tol=5,
                           disconnectedLV_area_diff_tol=0.05, disconnectedLV_x_diff_tol=20):
    """
    Processes the Left Ventricle (LV) segmentation to remove extra ventricular tissue.

    The crude segmentation of the left and right LVs may appear connected when they
    are enlarged (e.g., NPH and AD) or may appear disconnected (mostly in HC and PTE).

    Parameters
    ----------
    clean_ventricles : numpy.ndarray
        A 2D binary or labeled array representing the initial LV segmentation.
    suppress_blob_area_ratio : float, optional
        Threshold ratio to suppress blobs occupying less than this fraction of
        the maximum blob area. Default is 0.3 (30%).
    disconnectedLV_height_diff_tol : int or float, optional
        Maximum allowed difference in the vertical span (height) between two blobs
        to be considered valid disconnected LVs. Default is 5.
    disconnectedLV_area_diff_tol : float, optional
        Minimum required area ratio of the smaller blob relative to the second blob
        for the two blobs to be considered valid disconnected LVs. Default is 0.05 (5%).
    disconnectedLV_x_diff_tol : int or float, optional
        Maximum allowed horizontal distance from the image center for valid disconnected LV blobs.
        Default is 20.

    Returns
    -------
    disconnected_case : bool
        True if the algorithm determines the segmentation represents disconnected
        LVs, False otherwise.
    processed_blobs : numpy.ndarray
        A 2D binary array of the cleaned and processed ventricle segmentation.

    Raises
    ------
    ValueError
        If no segmented area remains after initial suppression, or if the algorithm
        fails to suppress non-ventricular masses.
    """
    blobs_labels = measure.label(clean_ventricles, background=0)

    # Grab counts of all distinct blobs (except the background)
    blob_sums_unprocessed = np.bincount(blobs_labels.flatten())[1:]

    # For cases with more than 2 blobs excluding the background
    if len(blob_sums_unprocessed) > 2:
        # Suppress blobs which occupy less than the specified area ratio of the largest blob.
        max_blob_area = np.max(blob_sums_unprocessed)
        suppress_labels = [ind + 1 for ind in np.where(blob_sums_unprocessed / max_blob_area < suppress_blob_area_ratio)[0]]

        # Suppress labels for non-ventricular tissue
        for sl in suppress_labels:
            blobs_labels[blobs_labels == sl] = 0

    # Make sure there is at least 1 non-background connected component
    if len(np.unique(blobs_labels)) <= 1:
        raise ValueError("No segmented area remains, check processing up to here.")

    processed_blobs = blobs_labels

    # If there are more than 3 non-background disconnected components, keep the largest 3
    if len(np.unique(blobs_labels)) > 3:
        top_3_args = np.argsort(np.bincount(blobs_labels.flatten())[1:])[::-1][:3]

        processed_blobs_1 = np.where(blobs_labels == top_3_args[0] + 1, blobs_labels, 0)
        pr1, _ = processed_blobs_1.nonzero()

        processed_blobs_2 = np.where(blobs_labels == top_3_args[1] + 1, blobs_labels, 0)
        pr2, _ = processed_blobs_2.nonzero()

        processed_blobs_3 = np.where(blobs_labels == top_3_args[2] + 1, blobs_labels, 0)
        pr3, _ = processed_blobs_3.nonzero()

        # Compare the y-direction medians to exclude the 3V (which sits lower)
        closest_inds = np.argmin([
            np.abs(np.median(pr1) - np.median(pr2)),
            np.abs(np.median(pr1) - np.median(pr3)),
            np.abs(np.median(pr2) - np.median(pr3))
        ])

        if closest_inds == 0:
            processed_blobs = processed_blobs_1 + processed_blobs_2
        elif closest_inds == 1:
            processed_blobs = processed_blobs_1 + processed_blobs_3
        else:
            processed_blobs = processed_blobs_2 + processed_blobs_3

    # There should ideally be no non-ventricular tissue here
    if len(np.unique(processed_blobs)) > 3:
        raise ValueError("Failed to suppress non-ventricular mass. Check segmentation up to here.")

    # Start by assuming a connected LV and an irrelevant non-background component
    disconnected_case = False

    # Make the blob labels contiguous using standard NumPy mapping
    rem_labels = np.unique(processed_blobs)
    mapped_blobs = np.zeros_like(processed_blobs)
    for new_val, old_val in enumerate(rem_labels):
        mapped_blobs[processed_blobs == old_val] = new_val
    processed_blobs = mapped_blobs

    # Check for any non-ventricular tissue based on blob height and laterality
    _, pc = processed_blobs.nonzero()
    im_cen = (np.max(pc) + np.min(pc)) // 2

    # If there are exactly 2 non-background blobs
    if len(np.unique(processed_blobs)) == 3:
        counts = np.bincount(processed_blobs.flatten())

        smallest_blob_arg = np.argsort(counts)[0]
        smallest_blob = np.where(processed_blobs == smallest_blob_arg, 1, 0)
        sb_r, sb_c = smallest_blob.nonzero()
        smallest_blob_height = (np.min(sb_r) + np.max(sb_r)) // 2
        smallest_blob_cen_x = (np.min(sb_c) + np.max(sb_c)) // 2

        mid_blob_arg = np.argsort(counts)[1]
        mid_blob = np.where(processed_blobs == mid_blob_arg, 1, 0)
        mb_r, mb_c = mid_blob.nonzero()
        mid_blob_height = (np.min(mb_r) + np.max(mb_r)) // 2
        mid_blob_cen_x = (np.min(mb_c) + np.max(mb_c)) // 2

        # CONDITION 1: Difference in vertical span of the two blobs
        nv_mass = np.abs(smallest_blob_height - mid_blob_height) > disconnectedLV_height_diff_tol

        # CONDITION 2: Smallest blob area ratio check
        area_check = np.sum(smallest_blob) < (disconnectedLV_area_diff_tol * np.sum(mid_blob))

        # CONDITION 1 AND CONDITION 2 should not hold, and both blobs should be centered: Declare disconnected LVs
        centered_small = np.abs(smallest_blob_cen_x - im_cen) <= disconnectedLV_x_diff_tol
        centered_mid = np.abs(mid_blob_cen_x - im_cen) <= disconnectedLV_x_diff_tol

        if centered_small and centered_mid and not nv_mass and not area_check:
            disconnected_case = True
        else:
            # Suppress non-ventricular tissue
            processed_blobs = np.where(processed_blobs == smallest_blob_arg, 0, processed_blobs)

    # Binarize the processed image
    processed_blobs = np.where(processed_blobs > 0, 1, 0)

    return disconnected_case, processed_blobs

## Contour tracing and angle computation

In [ ]:
def trace_ca_contour(lat_ventricles, coronal_plane, disconnected_case):
    """
    Traces the roof of the segmented lateral ventricles and calculates the
    callosal angle by fitting lines to the medial walls.

    Parameters
    ----------
    lat_ventricles : numpy.ndarray
        A 2D binary array representing the segmented lateral ventricles.
    coronal_plane : numpy.ndarray
        A 2D array representing the original coronal anatomical image, used
        for visualization.
    disconnected_case : bool
        Flag indicating whether the ventricles are disconnected (True) or
        connected as a single blob (False).

    Returns
    -------
    ca_angle : float
        The calculated interior corpus callosum (CA) angle in degrees.
    """
    # Centroid of the segmented image to help with stopping conditions
    M = cv.moments(np.uint8(lat_ventricles))
    cX = int(M["m10"] / M["m00"])
    cY = int(M["m01"] / M["m00"])

    # Connected case: single Moore contour, split left/right at midpoint
    if not disconnected_case:
        ven_contour = moore_contour(lat_ventricles)

        [vc_r, vc_c] = ven_contour.nonzero()

        ##### Left LV Processing
        l_stop_pt = int((min(vc_c) + max(vc_c)) // 2) - 2  # stop tracing the left LV contour at this point

        # Grab the left LV
        [lvc_r, lvc_c] = ven_contour[:, :l_stop_pt].nonzero()

        # If the left LV is not captured, move the end point to the image center
        if len(lvc_r) == 0:
            l_stop_pt = int((min(vc_c) + max(vc_c)) // 2)
            [lvc_r, lvc_c] = ven_contour[:, :l_stop_pt].nonzero()

        # Grab the top-most point of the left LV and the corresponding left-most point on that row
        lv_tp_y = min(lvc_r)
        lv_tp_x = min(lvc_c[lvc_r == lv_tp_y])

        # Start contour tracing SE/S/E from top-left of left LV to centroid
        row, col = lv_tp_y, lv_tp_x
        bot_con_row = cY
        left_pts, left_con_pts = trace_left_contour(row, col, bot_con_row, ven_contour)

        ##### Right LV Processing
        r_start_pt = int((min(vc_c) + max(vc_c)) // 2) + 2

        # Grab the right LV
        [rvc_r, rvc_c] = ven_contour[:, r_start_pt:].nonzero()

        # If the right LV is not captured, move the starting point to the image center
        if len(rvc_r) == 0:
            r_start_pt = int((min(vc_c) + max(vc_c)) // 2)
            [rvc_r, rvc_c] = ven_contour[:, r_start_pt:].nonzero()

        # Grab the top-most point of the right LV and the corresponding right-most point on that row
        rv_tp_y = min(rvc_r)
        rv_tp_x = max(rvc_c[rvc_r == rv_tp_y]) + r_start_pt

        # Start contour tracing SW/S/W from top-right of right LV to centroid
        row, col = rv_tp_y, rv_tp_x
        right_pts, right_con_pts = trace_right_contour(row, col, bot_con_row, ven_contour)

    else:
        # Disconnected case: separate Moore contours for left and right
        [lat_r, lat_c] = lat_ventricles.nonzero()

        left_ven_full = lat_ventricles.copy()
        left_ven_full[:, (min(lat_c) + max(lat_c)) // 2:] = 0
        left_ven_full = getLargestCC(measure.label(left_ven_full, background=0))

        right_ven_full = lat_ventricles.copy()
        right_ven_full[:, :(min(lat_c) + max(lat_c)) // 2] = 0
        right_ven_full = getLargestCC(measure.label(right_ven_full, background=0))

        left_ven_con = moore_contour(left_ven_full)
        right_ven_con = moore_contour(right_ven_full)
        ven_contour = left_ven_con + right_ven_con

        ##### Left LV Processing
        [lv_r, lv_c] = left_ven_con.nonzero()
        lv_tp_y = min(lv_r)
        lv_tp_x = min(lv_c[lv_r == lv_tp_y]) + 2

        row, col = lv_tp_y, lv_tp_x
        bot_con_row = cY

        left_pts, left_con_pts = trace_left_contour(row, col, bot_con_row, ven_contour)

        ##### Right LV Processing
        [rv_r, rv_c] = right_ven_con.nonzero()
        rv_tp_y = min(rv_r)
        rv_tp_x = max(rv_c[rv_r == rv_tp_y]) - 2

        row, col = rv_tp_y, rv_tp_x
        right_pts, right_con_pts = trace_right_contour(row, col, bot_con_row, ven_contour)

    ## Clean up left LV contour points — filter to middle 60% of vertical range
    left_min_row, left_max_row = left_con_pts[0][1], left_con_pts[-1][1]
    left_y_range = left_max_row - left_min_row
    left_low_thresh = left_max_row - 0.2 * left_y_range
    left_high_thresh = left_min_row + 0.2 * left_y_range
    left_con_cand_pts = [(x, y) for x, y in left_con_pts if (y >= left_high_thresh) & (y <= left_low_thresh)]
    if len(left_con_cand_pts) < 5:
        try:
            left_con_cand_pts = left_pts[:np.where(np.diff([y for x, y in left_pts]) > 0)[0][-1]]
        except IndexError:
            left_con_cand_pts = left_pts

    ## Clean up right LV contour points — filter to middle 60% of vertical range
    right_min_row, right_max_row = right_con_pts[0][1], right_con_pts[-1][1]
    right_y_range = right_max_row - right_min_row
    right_low_thresh = right_max_row - 0.2 * right_y_range
    right_high_thresh = right_min_row + 0.2 * right_y_range
    right_con_cand_pts = [(x, y) for x, y in right_con_pts if (y >= right_high_thresh) & (y <= right_low_thresh)]
    if len(right_con_cand_pts) < 5:
        try:
            right_con_cand_pts = right_pts[:np.where(np.diff([y for x, y in right_pts]) > 0)[0][-1]]
        except IndexError:
            right_con_cand_pts = right_pts

    ## Fit lines to the traced contour points and calculate angle between them
    ca_angle = fit_lines(left_con_cand_pts, right_con_cand_pts)

    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(coronal_plane, cmap='gray')
    [ax.scatter(x, y, marker='x', color='w') for (x, y) in left_con_cand_pts]
    [ax.scatter(x, y, marker='x', color='w') for (x, y) in right_con_cand_pts]
    plt.show()

    return ca_angle

## Main pipeline: process_calculate_ca()

In [ ]:
def process_calculate_ca(img_arr, pc_index, params, visualize=False):
    """Segments lateral ventricles from coronal slice at PC and computes callosal angle.

    For MRI: img_arr is the 3D numpy array (already rotated, no windowing).
    pc_index is the voxel index of PC in the volume.

    Parameters
    ----------
    img_arr : numpy.ndarray
        3D T1w MRI volume as a numpy array (already reoriented, no HU windowing).
        For OASIS: shape is (Z, Y, X); coronal slices are arr[:, y, :].
        For ABCD: extract_coronal_view() should be applied before passing.
    pc_index : tuple of int
        Voxel-space (x, y, z) coordinates of the Posterior Commissure.
        The coronal slice is extracted at index pc_index[1] (the Y voxel coordinate).
    params : dict
        Processing parameters dict (see ca_processing_params).
    visualize : bool, optional
        If True, displays intermediate segmentation steps. Default is False.

    Returns
    -------
    ca_angle : float
        The calculated interior callosal angle in degrees. Returns np.nan if
        the extraction or contour tracing pipeline fails.
    """
    # Unpack processing parameters
    gauss_blur_kernel_size = params['gauss_blur_kernel_size']
    adaptive_thresh_size = params['adaptive_thresh_size']
    adaptive_thresh_C = params['adaptive_thresh_C']
    above_pc_mask_threshold = params['above_pc_mask_threshold']
    middle_roi_xlims = params['middle_roi_xlims']
    middle_roi_upper_ylim = params['middle_roi_upper_ylim']
    smoothing_factor = params['smoothing_factor']
    dec_smooth_factor_max_thresh = params['dec_smooth_factor_max_thresh']
    initial_smooth_ven_fullbrain_ratio = params['initial_smooth_ven_fullbrain_ratio']
    smoothed_ven_fullbrain_ratio = params['smoothed_ven_fullbrain_ratio']
    median_blur_kernel_size = params['median_blur_kernel_size']

    # Step 1: Extract coronal slice at PC y-coordinate
    # pc_index is (x, y, z) in voxel space; the coronal plane is at index y
    pl = int(pc_index[1])
    coronal_plane = img_arr[:, pl, :]

    # image z index of the PC's location (used for above-PC ROI masking)
    # In array indexing: rows = Z axis, so PC z-row = img_arr.shape[0] - pc_index[2]
    pt = int(img_arr.shape[0] - pc_index[2])

    # Step 2: Gaussian blur
    coronal_gauss_blurred = cv.GaussianBlur(
        normalize_to_uint8(coronal_plane),
        (gauss_blur_kernel_size, gauss_blur_kernel_size),
        cv.BORDER_DEFAULT
    )

    # Step 2a: Otsu threshold — global intensity split (replaces HU windowing from CT)
    otsu_thresh = filters.threshold_otsu(coronal_gauss_blurred)
    coronal_bin = coronal_gauss_blurred > otsu_thresh

    # Step 3: Floodfill for brain mask
    im_th = coronal_bin.copy()
    h, w = im_th.shape[:2]
    mask = np.zeros((h + 2, w + 2), np.uint8)
    im_floodfill = im_th.copy()
    res = cv.floodFill(np.uint8(im_floodfill), mask, (0, 0), 255)
    full_brain_mask = (1 - res[2]).copy()
    full_brain_mask = full_brain_mask[1:h + 1, 1:w + 1]

    # Step 2b: Adaptive threshold
    coronal_adapt_thresh = cv.adaptiveThreshold(
        coronal_gauss_blurred, 1, cv.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv.THRESH_BINARY, adaptive_thresh_size, adaptive_thresh_C
    )
    coronal_adapt_thresh_bg_suppressed = np.where((1 - full_brain_mask) > 0, 0, coronal_adapt_thresh)
    coronal_brain_mask = np.where(coronal_adapt_thresh_bg_suppressed, 1, 0)

    # Step 4: Diagonal kernel morphological opening to break choroid plexus
    open_coronal_brain_mask = cv.morphologyEx(
        np.uint8(cv.morphologyEx(
            np.uint8(coronal_brain_mask),
            cv.MORPH_OPEN,
            np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], 'uint8'),
            iterations=1
        )),
        cv.MORPH_OPEN,
        np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]], 'uint8'),
        iterations=1
    )

    # Step 5: Combine adaptive + Otsu ventricles
    ventricles = np.logical_or(
        np.where(full_brain_mask - open_coronal_brain_mask > 0, 1, 0),
        np.where((1 - coronal_bin) * full_brain_mask > 0, 1, 0)
    )

    # Step 6: Mask above PC only — retain segmentation superior to PC
    above_pc_mask = np.zeros(ventricles.shape)
    above_pc_mask[0:pt - above_pc_mask_threshold, :] = 1
    ventricles = np.where(ventricles * above_pc_mask > 0, 1, 0)

    # Step 7: Clean up with middle ROI + largest CC
    M = cv.moments(np.uint8(full_brain_mask))
    bcX = int(M["m10"] / M["m00"])
    bcY = int(M["m01"] / M["m00"])

    middle_roi = np.zeros(ventricles.shape)
    middle_roi[bcY - middle_roi_upper_ylim:bcY, bcX - middle_roi_xlims:bcX + middle_roi_xlims] = 1
    clean_ven_1 = np.logical_or(middle_roi, ventricles)
    blobs_labels = measure.label(clean_ven_1, background=0)
    largest_cc_ven = getLargestCC(blobs_labels)
    [row_v, col_v] = (1 - largest_cc_ven).nonzero()
    ventricles_final = ventricles.copy()
    ventricles_final[row_v, col_v] = 0

    [vr, vc] = ventricles_final.nonzero()
    lv_roi = np.zeros(ventricles_final.shape)
    iqr_row = np.percentile(vr, 75) - np.percentile(vr, 25)
    upper_thresh_row = int(np.round((np.percentile(vr, 75) + 0.5 * iqr_row)))
    lv_roi[:upper_thresh_row, :] = 1

    # Step 8: Adaptive smoothing — reduce kernel until ratio meets threshold
    if (np.sum(ventricles_final) / np.sum(full_brain_mask)) > initial_smooth_ven_fullbrain_ratio:
        ventricles_final_smooth_0 = cv.medianBlur(np.uint8(ventricles_final), median_blur_kernel_size)
    else:
        ventricles_final_smooth_0 = ventricles_final

    dec_thresh = 0

    while True:
        lat_ventricles_smoothed = cv.medianBlur(
            np.uint8(lv_roi * ventricles_final_smooth_0),
            smoothing_factor - dec_thresh
        )
        if np.sum(lat_ventricles_smoothed) / np.sum(full_brain_mask) >= smoothed_ven_fullbrain_ratio:
            break
        else:
            dec_thresh = dec_thresh + 2  # reducing odd smoothing factors by 2
            if dec_thresh > dec_smooth_factor_max_thresh:
                print('Ventricles too small, not smoothed')
                lat_ventricles_smoothed = lv_roi * ventricles_final_smooth_0
                break

    if visualize:
        plt.imshow(coronal_plane, cmap='gray')
        plt.title('Coronal Plane at PC')
        plt.scatter(pc_index[0], pt, marker='x', color='r', label='PC')
        plt.legend()
        plt.show()
        plt.imshow(coronal_gauss_blurred, cmap='gray')
        plt.title('Coronal Plane - Gauss blurred')
        plt.show()
        plt.imshow(coronal_bin, cmap='gray')
        plt.title('Otsu thresholding')
        plt.show()
        plt.imshow(coronal_adapt_thresh, cmap='gray')
        plt.title('Adaptive thresholding')
        plt.show()
        plt.imshow(coronal_brain_mask, cmap='gray')
        plt.title('Adaptive threshold background suppressed')
        plt.show()
        plt.imshow(open_coronal_brain_mask, cmap='gray')
        plt.title('Open mask - to eliminate choroid plexus')
        plt.show()
        plt.imshow(full_brain_mask, cmap='gray')
        plt.title('Full brain mask - from floodfill')
        plt.show()
        plt.imshow(ventricles, cmap='gray')
        plt.title('Ventricles (above PC)')
        plt.show()
        plt.imshow(clean_ven_1, cmap='gray')
        plt.title('Ventricles connected via middle ROI')
        plt.show()
        plt.imshow(largest_cc_ven, cmap='gray')
        plt.title('LV extracted as one CC')
        plt.show()
        plt.imshow(ventricles_final, cmap='gray')
        plt.title('LV-v0')
        plt.show()
        plt.imshow(ventricles_final_smooth_0, cmap='gray')
        plt.title('LV-v0 (initial smooth)')
        plt.show()
        plt.imshow(lat_ventricles_smoothed, cmap='gray')
        plt.title('LV smoothed')
        plt.show()

    # Step 9: num_connected_comps_cv for cleanup
    lat_ventricles = None
    try:
        disconnected_case, lat_ventricles = num_connected_comps_cv(lat_ventricles_smoothed)
    except ValueError as e:
        print(f"[Warning] num_connected_comps_cv failed initially ({e}). Attempting fallback smoothing...")
        try:
            disconnected_case, lat_ventricles = num_connected_comps_cv(np.uint8(lv_roi * ventricles_final))
        except Exception as fallback_error:
            print(f"[Error] Fallback connected component step also failed: {fallback_error}")
            ca_angle = np.nan
            lat_ventricles = None
    except Exception as e:
        print(f"[Critical Error] Unexpected failure in num_connected_comps_cv: {e}")
        ca_angle = np.nan
        lat_ventricles = None

    # Step 10: trace_ca_contour for angle
    if lat_ventricles is not None:
        try:
            ca_angle = trace_ca_contour(lat_ventricles, coronal_plane, disconnected_case)
        except IndexError as e:
            # Step 11: Error handling with fallback smoothing
            print(f"[Warning] Contour tracing failed due to over-smoothing ({e}). Applying median blur fallback...")
            try:
                smoothed_fallback = cv.medianBlur(np.uint8(lat_ventricles), median_blur_kernel_size)
                ca_angle = trace_ca_contour(smoothed_fallback, coronal_plane, disconnected_case)
            except Exception as fallback_error:
                print(f"[Error] Fallback contour tracing failed: {fallback_error}")
                ca_angle = np.nan
        except Exception as e:
            print(f"[Critical Error] Unexpected failure in trace_ca_contour: {e}")
            ca_angle = np.nan
    else:
        print("[Skipped] contour tracing bypassed because ventricles extraction failed in num_connected_comps_cv")
        ca_angle = np.nan

    return ca_angle

## AC/PC coordinate loading

The Callosal Angle is measured on the **coronal slice at the Posterior Commissure (PC)** level.

- **OASIS**: The atlas-registered T88 volumes share a fixed AC/PC in voxel space (`OASIS_PC_VOXEL` from `mri_utils`).
- **ABCD**: Each subject has a per-scan `*_acpc_coords.txt` file produced by Charlie's ART script. Physical coordinates are converted to voxel indices via SimpleITK.

In [ ]:
def get_pc_for_subject(subject_id, scan_path, dataset, acpc_txt_glob=None):
    """Returns the PC voxel index (x, y, z) for a given subject.

    For OASIS: uses the hardcoded OASIS_PC_VOXEL atlas coordinate.
    For ABCD: loads the per-subject AC/PC txt file and converts physical
    coordinates to voxel indices using the scan's SimpleITK image metadata.

    Parameters
    ----------
    subject_id : str
        Subject identifier string.
    scan_path : str
        Path to the subject's NIfTI scan file.
    dataset : str
        One of 'oasis1', 'abcd', or 'custom'.
    acpc_txt_glob : str or None
        Glob pattern (relative to the scan's directory) for AC/PC txt files.
        Required for ABCD and custom datasets. Not used for OASIS.

    Returns
    -------
    pc_voxel : tuple of int
        Voxel-space PC coordinates as (x, y, z).
    """
    if dataset == "oasis1":
        # OASIS T88 atlas: all subjects share a fixed PC voxel coordinate
        return OASIS_PC_VOXEL  # (85, 87, 73)

    # ABCD / custom: load per-subject AC/PC txt and convert physical -> voxel
    scan_dir = os.path.dirname(scan_path)
    txt_files = glob.glob(os.path.join(scan_dir, acpc_txt_glob or "*_acpc_coords.txt"), recursive=True)

    if len(txt_files) == 0:
        raise FileNotFoundError(
            f"No AC/PC txt file found for subject {subject_id} in {scan_dir}. "
            f"Expected pattern: {acpc_txt_glob}"
        )

    _, pc_physical = load_acpc_from_txt(txt_files[0])

    # Convert physical -> voxel using the scan's SimpleITK metadata
    img = sitk.ReadImage(scan_path)
    img_rotated = rotate_image(img)
    pc_voxel_continuous = img_rotated.TransformPhysicalPointToContinuousIndex(pc_physical)
    pc_voxel = tuple(int(np.round(v)) for v in pc_voxel_continuous)

    return pc_voxel

## Discover available scans

In [ ]:
scan_files = sorted(glob.glob(os.path.join(DATA_DIR, SCAN_GLOB), recursive=True))

scan_df = pd.DataFrame({
    "subject_id": [SUBJECT_ID_FROM_PATH(p) for p in scan_files],
    "scan_path": scan_files,
})
print(f"Found {len(scan_df)} scans in {DATA_DIR}")
scan_df.head()

## Test on a single subject first

Run the full pipeline on one scan to verify everything works before batch processing.

In [ ]:
# Pick first subject
test_path = scan_df.iloc[0]['scan_path']
test_id = scan_df.iloc[0]['subject_id']
print(f"Testing on: {test_id}")
print(f"Path: {test_path}")

# Load, rotate, and convert to array
img = sitk.ReadImage(test_path)
img_rotated = rotate_image(img)
img_arr = sitk.GetArrayFromImage(img_rotated)
print(f"Array shape: {img_arr.shape}")
print(f"Intensity range: [{img_arr.min()}, {img_arr.max()}]")

# Get PC voxel index
pc_voxel = get_pc_for_subject(test_id, test_path, DATASET, ACPC_TXT_GLOB)
print(f"PC voxel index: {pc_voxel}")

# Show the coronal slice at PC
pl = int(pc_voxel[1])
coronal_at_pc = img_arr[:, pl, :]
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_title(f"Coronal slice at PC (y={pl}) — {test_id}")
ax.imshow(coronal_at_pc, cmap='gray')
plt.show()

In [ ]:
# Run pipeline on single subject with visualization
ca_angle_test = process_calculate_ca(img_arr, pc_voxel, ca_processing_params, visualize=True)
print(f"\nCallosal Angle for {test_id}: {ca_angle_test:.2f} degrees")

## Batch processing — all subjects

Run the same pipeline on all discovered scans with visualization off.

In [ ]:
ca_results = []

for _, row in scan_df.iterrows():
    subject_id = row['subject_id']
    vol_path = row['scan_path']

    try:
        img = sitk.ReadImage(vol_path)
        img_rotated = rotate_image(img)
        img_arr = sitk.GetArrayFromImage(img_rotated)

        pc_voxel = get_pc_for_subject(subject_id, vol_path, DATASET, ACPC_TXT_GLOB)

        ca_angle = process_calculate_ca(img_arr, pc_voxel, ca_processing_params, visualize=False)

        ca_results.append({"subject_id": subject_id, "ca_angle": ca_angle})
        print(f"{subject_id}: CA = {ca_angle:.2f} degrees" if not np.isnan(ca_angle) else f"{subject_id}: CA = NaN")

    except Exception as e:
        print(f"{subject_id}: ERROR — {e}")
        ca_results.append({"subject_id": subject_id, "ca_angle": np.nan})

ca_df = pd.DataFrame(ca_results)
print(f"\nProcessed {len(ca_df)} subjects ({ca_df['ca_angle'].notna().sum()} successful)")

## Results and histogram

In [ ]:
print(f"CA summary (n={ca_df['ca_angle'].notna().sum()}):")
print(ca_df['ca_angle'].describe().round(2))
print(f"\nNaN count: {ca_df['ca_angle'].isna().sum()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of CA values
valid_ca = ca_df['ca_angle'].dropna()
axes[0].hist(valid_ca, bins=20, edgecolor='black')
axes[0].set_xticks(np.arange(20, 160, 10))
axes[0].set_xlabel('Callosal Angle (degrees)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Callosal Angle Distribution (n={len(valid_ca)})')

# Flag extreme values
min_ca_lim = 40
max_ca_lim = 150
extreme = ca_df[(ca_df['ca_angle'] <= min_ca_lim) | (ca_df['ca_angle'] >= max_ca_lim)]
axes[1].axis('off')
axes[1].set_title(f'Extreme values (CA <= {min_ca_lim} or CA >= {max_ca_lim})')
if len(extreme) > 0:
    axes[1].text(0.05, 0.95, extreme.to_string(index=False), transform=axes[1].transAxes,
                 fontsize=10, verticalalignment='top', fontfamily='monospace')
else:
    axes[1].text(0.5, 0.5, 'No extreme values detected', transform=axes[1].transAxes,
                 ha='center', va='center', fontsize=12)

plt.tight_layout()
plt.show()

print(f"\nSubjects with extreme CA values (CA <= {min_ca_lim} or CA >= {max_ca_lim}):")
print(extreme if len(extreme) > 0 else "None")

In [ ]:
output_path = os.path.join(OUTPUT_DIR, f"callosal_angle_{DATASET}.csv")
ca_df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")